In [9]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

True

In [2]:
with open("blood_work.png","rb") as f:
    image_b64= base64.b64encode(f.read()).decode()

image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzka'

In [3]:
llm= ChatGroq( model="qwen/qwen3.8-27b")

message=HumanMessage(content=[
    {"type":"image_url","image_url":{"url":f"data:image/png;base64,{image_b64}"}},
    {"type":"text","text":"This is a blood work report. Please summarize the results and provide any recommendations."}
])

response =llm.invoke([message])
print(response.content)

Based on the blood work report provided for **Rajesh Sharma, Age 48, Male**, dated **May 7, 2026**, here is a summary of the results and recommendations:

---

### **1. Complete Blood Count (CBC)**

- **Hemoglobin**: 15.1 g/dL → *Normal* (Range: 13.5–17.5)
- **Hematocrit**: 44% → *Normal* (Range: 41–53%)
- **WBC (White Blood Cells)**: 6.8 x 10³/µL → *Normal* (Range: 4.5–11.0)
- **Platelets**: 220 x 10³/µL → *Normal* (Range: 150–400)

✅ **Interpretation**: All CBC values are within normal limits. No signs of anemia, infection, or clotting disorders.

---

### **2. Lipid Panel**

- **Total Cholesterol**: 238 mg/dL → *High* (Normal: <200)
- **LDL Cholesterol (“Bad” Cholesterol)**: 162 mg/dL → *High* (Normal: <100)
- **HDL Cholesterol (“Good” Cholesterol)**: 36 mg/dL → *Low* (Normal: >40)
- **Triglycerides**: 188 mg/dL → *Borderline High* (Normal: <150)

⚠️ **Interpretation**: This lipid profile indicates **dyslipidemia** — specifically:
- Elevated total and LDL cholesterol (increased card

In [10]:
@tool
def get_dite_recommendations(condition: str) -> dict:
    """" given a health condition, return diet plan. condition must be one of the following: diabetes, hypertension, high cholesterol, obesity, celiac disease, kidney disease, liver disease, heart disease, thyroid disorders, anemia"""
    diet_plans = {
        "diabetes":{
            "eat": ["whole grains", "vegetables", "fruits", "lean proteins", "legumes"],
            "avoid": ["sugary foods", "refined carbs", "sweetened beverages", "processed foods"]
        },
        "hypertension":{
            "eat": ["fruits", "vegetables", "whole grains", "low-fat dairy", "lean proteins"],
            "avoid": ["salt", "processed foods", "saturated fats", "alcohol"]
        },
        "high cholesterol":{
            "eat": ["oats", "barley", "nuts", "seeds", "fatty fish"],
            "avoid": ["trans fats", "saturated fats", "processed foods", "red meat"]
        },
        "normal":{
            "eat": ["balanced diet with fruits, vegetables, whole grains, lean proteins"],
            "avoid": ["excessive sugar, salt, and unhealthy fats"]
        }
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [25]:
SYSTEM_PROMPT = """You are a helpful assistant that provides diet recommendations based on health conditions.
for the input blood work image, extract the number and the normal range, then categorize the condition as one of the following: diabetes,hypertension,high cholesterol,normal.
 Then call approiate method  to get diet recommendations for the condition. Finally,  provide diet recommendations in a clear and concise manner."""
llm2= ChatGroq( model="qwen/qwen3.8-27b")


diet_agent = create_agent(
    llm2,
    tools=[get_dite_recommendations],
    system_prompt=SYSTEM_PROMPT
)

In [26]:

result = diet_agent.invoke({"messages": [HumanMessage(content=[
    {"type":"image_url","image_url":{"url":f"data:image/png;base64,{image_b64}"}},
    {"type":"text","text":"Analyze this blood work report and suggest a diet plan based on the results."},
])]
})

print(result["messages"][-1].content)

Based on the blood work report for **Rajesh Sharma (Age 48, Male)**, here is the analysis and diet plan:

**1. Blood Work Analysis & Categorization**

*   **Glucose (Fasting):** 92 mg/dL (Normal: 70–99) ✅ *Normal*
*   **HbA1c:** 5.3% (Normal: < 5.7%) ✅ *Normal*
*   **Blood Pressure:** Not provided, but metabolic markers are normal.
*   **Lipid Panel:**
    *   **Total Cholesterol:** 238 mg/dL (Normal: < 200) ❌ *High*
    *   **LDL Cholesterol:** 162 mg/dL (Normal: < 100) ❌ *High*
    *   **HDL Cholesterol:** 36 mg/dL (Normal: > 40) ❌ *Low*
    *   **Triglycerides:** 188 mg/dL (Normal: < 150) ❌ *High*

**Categorization:** **High Cholesterol**

**2. Diet Recommendations for High Cholesterol**

To help lower LDL and Triglycerides while raising HDL, the following dietary changes are recommended:

**Foods to Eat (Include in diet):**
*   **Oats:** High in soluble fiber, which helps lower LDL cholesterol.
*   **Barley:** Another excellent source of soluble fiber.
*   **Nuts:** Such as almonds